# Generating outliers

**Outliers** are *appended* data points: the generator samples new
rows that look extreme relative to the given data, and stacks them **after**
the original rows — so `Xt` is *larger* than `X`. Use outliers to test
how pipelines behave when rare, extreme observations appear: robust
statistics, anomaly detection, scaling, and model fitting all react
very differently to them.

This notebook shows the **two levers** available in badgers:

1. **Acting on the observation process** — sample outliers directly
   from distributions fitted to the data (`ZScoreSamplingGenerator`,
   `HypersphereSamplingGenerator`, histogram and low-density samplers).
   These generators *do not use a DGP*.
2. **Acting on the data-generating process (DGP)** — perturb a node of
   a *fitted causal model*; the perturbation **propagates** to the
   descendants, so the outliers are not just extreme — they are
   *structurally consistent* extreme events
   (`CausalOutlierPropagationGenerator`).

Labels: the generated rows are the *last* `n_outliers` rows of `Xt`;
the boolean array is exposed via `get_outliers_labels()` (labels do
**not** ride in the deprecated `yt` return slot).


## Goal

Generate out-of-distribution points in tabular data — sampled directly from the data's own geometry (Part A) or propagated along a causal graph (Part B) — and identify which rows are outliers.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import networkx as nx
import pandas as pd

from numpy.random import default_rng
from sklearn.datasets import make_blobs

from badgers.generators.tabular_data.outliers.distribution_sampling import (
    ZScoreSamplingGenerator, HypersphereSamplingGenerator,
)
from badgers.generators.tabular_data.outliers.low_density_sampling import (
    IndependentHistogramsGenerator, HistogramSamplingGenerator, LowDensitySamplingGenerator,
)
from badgers.generators.tabular_data.outliers import StructuredOutlierGenerator

rng = default_rng(seed=0)


In [ ]:
def plot_outliers(X, Xt, labels, cls=None):
    """
    Plot original data and the transformed data (outliers marked).
    labels = boolean outlier labels (get_outliers_labels());
    cls = optional class vector for coloring the original data.
    """
    outliers = Xt[labels]

    fig, axes = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(8,4))
    if cls is not None:
        for i in range(len(np.unique(cls))):
            ix = np.where(cls == i)
            axes[0].scatter(X[ix,0],X[ix,1], c = f'C{i}', label = f'{i}')
            axes[1].scatter(X[ix,0],X[ix,1], c = f'C{i}', label = f'{i}')
    else:
        axes[0].scatter(X[:,0], X[:,1], c='C0', label='original')
        axes[1].scatter(X[:,0], X[:,1], c='C0', label='original')
    axes[1].scatter(outliers[:,0],outliers[:,1], marker = 'x', c = f'black', label = 'outliers')
    axes[0].set_title('Original')
    axes[1].set_title('Transformed')
    axes[1].legend(ncol=1, bbox_to_anchor=(1, 1))
    plt.tight_layout()
    return fig, axes


## Load and prepare data

We first load an existing dataset from `sklearn.datasets`.


In [ ]:
X, y = make_blobs(centers=4, random_state=0, cluster_std=0.5)
cls = y.astype(int)   # explicit class vector for plotting
print(X.shape)

## Part A — Acting on the observation process (no DGP)

These generators sample outlier rows from distributions or densities
derived directly from the data. No causal model is involved; the
outliers are *marginal* extremes, unrelated to any structure between
the columns.

### Generate outliers using z-score sampling

`ZScoreSamplingGenerator` generates points whose z-score (distance to
the mean, in units of the standard deviation) exceeds 3 — far out on
the tails of the empirical distribution.

In [ ]:
gen = ZScoreSamplingGenerator(random_generator=rng)
Xt, _ = gen.generate(X.copy(), None, n_outliers=25)   # y=None (deprecated slot)
labels = gen.get_outliers_labels()
print('appending:', X.shape, '->', Xt.shape, '| outlier rows:', int(labels.sum()))

In [ ]:
fig, axes = plot_outliers(X, Xt, labels, cls)

### Generate outliers using hypersphere sampling

`HypersphereSamplingGenerator` generates points uniformly on a
hypersphere (a shell around the data) — extreme in every direction,
not only along single axes.

In [ ]:
gen = HypersphereSamplingGenerator(random_generator=rng)
Xt, _ = gen.generate(X.copy(), None, n_outliers=25)
labels = gen.get_outliers_labels()
print('appending:', X.shape, '->', Xt.shape, '| outlier rows:', int(labels.sum()))

In [ ]:
fig, axes = plot_outliers(X, Xt, labels, cls)

### Generate outliers using independent histogram sampling

`IndependentHistogramsGenerator` fits a histogram to each column
independently, then samples outliers from the empty (or rare) bins —
combinations that never occur in the data.

In [ ]:
gen = IndependentHistogramsGenerator(random_generator=rng)
Xt, _ = gen.generate(X.copy(), None, n_outliers=25)
labels = gen.get_outliers_labels()
print('appending:', X.shape, '->', Xt.shape, '| outlier rows:', int(labels.sum()))

In [ ]:
fig, axes = plot_outliers(X, Xt, labels, cls)

### Generate outliers using histogram sampling

`HistogramSamplingGenerator` works on the *joint* histogram of the
data — but only for low-dimensional datasets (the bin count explodes
with dimensionality).

In [ ]:
gen = HistogramSamplingGenerator(random_generator=rng)
Xt, _ = gen.generate(X.copy(), None, n_outliers=25)
labels = gen.get_outliers_labels()
print('appending:', X.shape, '->', Xt.shape, '| outlier rows:', int(labels.sum()))

In [ ]:
fig, axes = plot_outliers(X, Xt, labels, cls)

### Generate outliers using low density sampling

`LowDensitySamplingGenerator` samples points in the *low-density*
regions of the empirical distribution — between and beyond the
clusters, rather than at the extremes of single columns.

In [ ]:
gen = LowDensitySamplingGenerator(random_generator=rng)
Xt, _ = gen.generate(X.copy(), None, n_outliers=25)
labels = gen.get_outliers_labels()
print('appending:', X.shape, '->', Xt.shape, '| outlier rows:', int(labels.sum()))

In [ ]:
fig, axes = plot_outliers(X, Xt, labels, cls)

### Generate outliers in a reduced space (decomposition + sampling)

For higher-dimensional data, a first step is to **reduce the dimensionality**
(any sklearn transformer with `transform`/`inverse_transform`, e.g.
`PCA`, `KernelPCA`, `FastICA`) and sample outliers in the reduced
space: `StructuredOutlierGenerator` chains a decomposition and an
outlier sampler — outliers are sampled where the data *really* has
little support, in the informative subspace rather than the noisy one.

In [ ]:
from sklearn.decomposition import PCA, KernelPCA

# generate some data with higher dimensionality
X_hi, y_hi = make_blobs(n_features=10, centers=4, cluster_std=0.5, random_state=0)
cls_hi = y_hi.astype(int)

gen = StructuredOutlierGenerator(
    dimension_reduction_transformer=PCA(n_components=3),
    outlier_generator=ZScoreSamplingGenerator(),
)
Xt, _ = gen.generate(X_hi.copy(), None, n_outliers=25)
labels = gen.get_outliers_labels()
print('appending:', X_hi.shape, '->', Xt.shape, '| outlier rows:', int(labels.sum()))

In [ ]:
fig, axes = plot_outliers(X_hi, Xt, labels, cls_hi)

Any decomposition with `inverse_transform` works, e.g. a kernel PCA —
and the outlier sampler can be any sampler from this section:

In [ ]:
gen = StructuredOutlierGenerator(
    dimension_reduction_transformer=KernelPCA(n_components=3, kernel="rbf", fit_inverse_transform=True),
    outlier_generator=HypersphereSamplingGenerator(),
)
Xt, _ = gen.generate(X_hi.copy(), None, n_outliers=25)
labels = gen.get_outliers_labels()
print('appending:', X_hi.shape, '->', Xt.shape, '| outlier rows:', int(labels.sum()))

In [ ]:
fig, axes = plot_outliers(X_hi, Xt, labels, cls_hi)

Preprocessing stays out of the generator's responsibility: instead of
the built-in standardization + decomposition, a custom transformer can
be passed as `preprocessor` (anything with `transform`/`inverse_transform`,
e.g. a fitted sklearn pipeline):


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import FastICA

gen = StructuredOutlierGenerator(
    outlier_generator=ZScoreSamplingGenerator(),
    preprocessor=make_pipeline(RobustScaler(), FastICA(n_components=3, whiten="unit-variance")),
)
Xt, _ = gen.generate(X_hi.copy(), None, n_outliers=25)
labels = gen.get_outliers_labels()
print('appending:', X_hi.shape, '->', Xt.shape, '| outlier rows:', int(labels.sum()))


In [ ]:
fig, axes = plot_outliers(X_hi, Xt, labels, cls_hi)

## Part B — Acting on the data-generating process (DGP)

`CausalOutlierPropagationGenerator` perturbs a node of a *fitted causal
model* with `do()`-style intervention: the node is set to an extreme
value and the **descendants are recomputed** through the structural
equations. The result: outliers that are extreme *and structurally
consistent* — unlike Part A's marginal extremes, these respect the
relationships between the columns.

**The choice of perturbed node matters**, and the topology decides how far
the effect reaches — that is what the following examples show.


In [15]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import badgers
badgers.__version__

'0.0.14'

In [ ]:
# ============================================================
# Helper functions for the topology examples
# ============================================================

def generate_linear_data(graph, n_samples=100, rng=None, coefficients=None):
    """Generate data from a linear structural equation model defined by a DAG.

    Each edge (u -> v) has a coefficient w_uv. Node values are computed as:
        v = sum_{u in parents(v)} w_uv * u + noise
    where noise ~ N(0, 1).

    Parameters
    ----------
    graph : nx.DiGraph
        The causal DAG. Node names must be strings.
    n_samples : int
        Number of samples to generate.
    rng : np.random.Generator, optional
        Random number gen.
    coefficients : dict, optional
        Edge coefficients. Defaults to 1.0 for all edges.

    Returns
    -------
    X : np.ndarray of shape (n_samples, n_nodes)
    column_mapping : dict[str, int]
    """
    if rng is None:
        rng = np.random.default_rng(42)
    if coefficients is None:
        coefficients = {edge: 1.0 for edge in graph.edges()}

    nodes = list(graph.nodes())
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    n_nodes = len(nodes)

    X = np.zeros((n_samples, n_nodes))
    topo_order = list(nx.topological_sort(graph))

    for node in topo_order:
        idx = node_to_idx[node]
        parents = list(graph.predecessors(node))
        if not parents:
            # Exogenous node: pure noise
            X[:, idx] = rng.normal(size=n_samples)
        else:
            # Endogenous node: weighted sum of parents + noise
            X[:, idx] = rng.normal(size=n_samples)
            for parent in parents:
                parent_idx = node_to_idx[parent]
                coef = coefficients.get((parent, node), 1.0)
                X[:, idx] += coef * X[:, parent_idx]

    column_mapping = {node: i for i, node in enumerate(nodes)}
    return X, column_mapping

In [ ]:
def plot_data(X, y, column_names):
    """Scatter-matrix of the data with outliers highlighted in red.

    Parameters
    ----------
    X : np.ndarray of shape (n_samples, n_features)
    y : array-like or None
        Labels: "original" or "outliers". If None, all points are "original".
    column_names : list of str
        Names for each column.
    """
    is_outlier = np.asarray(y) == "outliers" if y is not None else np.zeros(len(X), dtype=bool)
    d = len(column_names)
    fig, axes = plt.subplots(d, d, figsize=(2.5 * d, 2.5 * d))
    for i in range(d):
        for j in range(d):
            ax = axes[i, j]
            if i == j:
                # diagonal: histogram of the original data
                ax.hist(X[~is_outlier, j], bins=25, color="C0", alpha=0.7)
                if is_outlier.any():
                    ax.hist(X[is_outlier, j], bins=25, color="C3", alpha=0.7)
                ax.set_ylabel("Count") if j == 0 else ax.set_yticklabels([])
            elif i > j:
                ax.set_visible(False)  # hide lower triangle
            else:
                # upper triangle: scatter + regression line on original data
                ax.scatter(X[~is_outlier, j], X[~is_outlier, i], color="C0", s=15, alpha=0.6)
                ax.scatter(X[is_outlier, j], X[is_outlier, i], color="C3", s=30, alpha=0.6)
                if (~is_outlier).sum() > 1:
                    coef = np.polyfit(X[~is_outlier, j], X[~is_outlier, i], 1)
                    xs = np.array([X[~is_outlier, j].min(), X[~is_outlier, j].max()])
                    ax.plot(xs, coef[0] * xs + coef[1], color="C0", linewidth=1, alpha=0.5)
            if i == d - 1:
                ax.set_xlabel(column_names[j])
            if j == 0:
                ax.set_ylabel(column_names[i])
    fig.suptitle("Data scatter-matrix (red = outliers)", y=1.02, fontsize=13)
    plt.tight_layout()
    return fig


In [ ]:
def draw_graph(graph, title=None):
    """Draw a DAG with fixed left-to-right node positions.

    Nodes are placed in topological generations: roots at x = 0, their
    descendants at x = 1, and so on; siblings are stacked vertically.
    This keeps the visual layout deterministic and consistent across
    all notebooks (no force-directed layout).
    """
    generations = {}  # node -> column index
    for node in nx.topological_sort(graph):
        parents = list(graph.predecessors(node))
        generations[node] = 0 if not parents else max(generations[p] for p in parents) + 1
    columns = {}
    pos = {}
    for node, col in generations.items():
        pos[node] = (col, columns.setdefault(col, 0))
        columns[col] += 1
    # center each column vertically
    height = max(columns.values())
    pos = {node: (x, y - (columns[x] - 1) / 2) for node, (x, y) in pos.items()}
    fig, ax = plt.subplots(1, figsize=(6, 2))
    nx.draw(graph, pos, ax=ax, node_color="lightblue", node_size=800, arrowsize=15)
    nx.draw_networkx_labels(graph, pos, ax=ax)
    if title:
        ax.set_title(title, fontsize=12)
    ax.axis("off")
    return fig


In [ ]:
# Part B: a fresh RNG stream (independent of Part A)
rng = np.random.default_rng(42)
print("Helper functions and RNG ready.")

### Chain: $X \rightarrow Y \rightarrow Z$

**Perturbing X** affects Y (direct child) and Z (indirect child).
**Perturbing Y** affects Z but *not* X. The propagation direction is
always downstream.

In [12]:
# --- Chain: X -> Y -> Z ---
chain = nx.DiGraph()
chain.add_edges_from([("X", "Y"), ("Y", "Z")])

coefs_chain = {("X", "Y"): 2.0, ("Y", "Z"): 0.5}
X_chain, mapping_chain = generate_linear_data(chain, n_samples=100, rng=rng, coefficients=coefs_chain)

print(f"X_chain shape: {X_chain.shape}")
print(f"column_mapping: {mapping_chain}")

X_chain shape: (100, 3)
column_mapping: {'X': 0, 'Y': 1, 'Z': 2}


#### Visualize graph and data

In [ ]:
# Draw the graph
draw_graph(chain, title="Chain: X → Y → Z")
plt.show()

# Pairplot of the original data
plot_data(X_chain, y=None, column_names=["X", "Y", "Z"])

#### Generate outliers (perturb X, then Y)

In [18]:
from badgers.core.dgp import OlsDGP
from badgers.generators.tabular_data.outliers.causal import CausalOutlierPropagationGenerator

In [ ]:
# Perturb X — do(X): X set directly, Y and Z recomputed through the DGP
dgp_chain = OlsDGP(chain, mapping_chain).fit(X_chain)
gen = CausalOutlierPropagationGenerator(random_generator=rng, dgp=dgp_chain)
Xt_x, _ = gen.generate(X_chain.copy(), y=None,
                                 perturbation_nodes=["X"],
                                 n_outliers=10)
labels_x = gen.get_outliers_labels()   # outlier labels do not ride in yt (returned y is None)

# Perturb Y — do(Y): X is exogenous (sampled normally), Y set directly, Z recomputed
dgp_chain_y = OlsDGP(chain, mapping_chain).fit(X_chain)
gen = CausalOutlierPropagationGenerator(random_generator=rng, dgp=dgp_chain_y)
Xt_y, _ = gen.generate(X_chain.copy(), y=None,
                                 perturbation_nodes=["Y"],
                                 n_outliers=10)
labels_y = gen.get_outliers_labels()

print(f"do(X): {Xt_x.shape[0]} rows ({Xt_x.shape[0] - 100} outliers)")
print(f"do(Y): {Xt_y.shape[0]} rows ({Xt_y.shape[0] - 100} outliers)")

#### Visualize generated data

In [ ]:
# do(X): outliers should be visible as red markers
plot_data(Xt_x, y=np.where(labels_x, 'outliers', 'original'), column_names=["X", "Y", "Z"])

In [ ]:
# do(Y): X is exogenous (not perturbed), Y and Z shift
plot_data(Xt_y, y=np.where(labels_y, 'outliers', 'original'), column_names=["X", "Y", "Z"])

### Fork (common cause): $C \rightarrow X, C \rightarrow Y$

**Perturbing the common cause C** moves *both* children X and Y — the
outliers appear correlated even though the perturbation hit a single
node.

In [28]:
# --- Fork: C -> X, C -> Y ---
fork = nx.DiGraph()
fork.add_edges_from([("C", "X"), ("C", "Y")])

coefs_fork = {("C", "X"): 1.5, ("C", "Y"): 0.8}
X_fork, mapping_fork = generate_linear_data(fork, n_samples=100, rng=rng, coefficients=coefs_fork)

#### Visualize graph and data

In [ ]:
# Draw the graph
draw_graph(fork, title="Fork: C → X, C → Y")
plt.show()

# Pairplot of the original data
plot_data(X_fork, y=None, column_names=["C", "X", "Y"])

#### Generate outliers

In [ ]:
dgp_fork = OlsDGP(fork, mapping_fork).fit(X_fork)
gen = CausalOutlierPropagationGenerator(random_generator=rng, dgp=dgp_fork)
Xt_fork, _ = gen.generate(X_fork.copy(), y=None,
                                 perturbation_nodes=["C"],
                                 n_outliers=10)
labels_fork = gen.get_outliers_labels()

In [ ]:
# verify the claim: perturbing the common cause correlates the two children
out_mask = labels_fork.astype(bool)
print('corr(X, Y) all rows    : %.3f' % np.corrcoef(Xt_fork[:, 1], Xt_fork[:, 2])[0, 1])
print('corr(X, Y) outlier rows: %.3f' % np.corrcoef(Xt_fork[out_mask, 1], Xt_fork[out_mask, 2])[0, 1])


#### Visualize generated data

In [ ]:
plot_data(Xt_fork, y=np.where(labels_fork, 'outliers', 'original'), column_names=["C", "X", "Y"])

### Sampling strategy comparison

The `within_distribution_sampler` and `out_of_distribution_sampler`
parameters control *how* the outlier values are drawn at the perturbation
node:

In [ ]:
from badgers.core.sampling import (
    create_out_of_distribution_sampler,
    UniformOutOfDistributionSampler,
)

# Use the chain graph from section 6.1
sampler_configs = [
    ("zscore", {}),
    ("hypersphere", {}),
    ("uniform", {"expansion": 0.5}),
]
n_outliers = 20

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes = axes.flatten()

for ax, (sampler_name, kwargs) in zip(axes, sampler_configs):
    if kwargs:
        sampler = create_out_of_distribution_sampler(sampler_name, **kwargs)
    else:
        sampler = sampler_name

    dgp = OlsDGP(chain, mapping_chain).fit(X_chain)
    gen = CausalOutlierPropagationGenerator(random_generator=rng, dgp=dgp)
    Xt, _ = gen.generate(
        X_chain.copy(), y=None,
        perturbation_nodes=["X"],
        n_outliers=n_outliers,
        out_of_distribution_sampler=sampler,
    )
    outliers = Xt[-n_outliers:]
    original = Xt[:-n_outliers]

    # Scatter: X vs Y, color by original/outlier
    ax.scatter(original[:, 0], original[:, 1], alpha=0.4, s=20, label="original", c="C0")
    ax.scatter(outliers[:, 0], outliers[:, 1], alpha=0.8, s=40, label="outliers", c="C3", edgecolors="black")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    label = sampler_name
    if kwargs:
        label = f"{sampler_name} ({', '.join(f'{k}={v}' for k, v in kwargs.items())})"
    ax.set_title(f'out_of_distribution_sampler="{label}"')
    ax.legend()
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.3)

plt.suptitle("Out-of-Distribution Sampler Comparison: X → Y (Chain Topology)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Key observations:**

- **`"zscore"`** (default): Outliers are pushed far from the mean ($\geq 3\sigma$), creating extreme but causally-consistent anomalies.
- **`"hypersphere"`**: Outliers form a spherical shell in standardized space, useful for testing detectors against uniformly-directional anomalies.
- **`"uniform"`**: Outliers spread uniformly beyond the observed range, producing diverse patterns across the full expanded space.

The within-distribution sampler (default `"normal"`) controls how exogenous
ancestor nodes are sampled — these stay within the observed distribution since
they are not directly perturbed.

## Summary

| Lever | Generator | Uses a DGP? | Propagates? | Effect on shape |
|---|---|---|---|---|
| observation process | `ZScoreSamplingGenerator` | no | no | appends `n_outliers` rows |
| observation process | `HypersphereSamplingGenerator` | no | no | appends `n_outliers` rows |
| observation process | `IndependentHistogramsGenerator` / `HistogramSamplingGenerator` / `LowDensitySamplingGenerator` | no | no | appends `n_outliers` rows |
| observation process + decomposition | `StructuredOutlierGenerator` | no | no | appends (sampling happens in the reduced space) |
| DGP | `CausalOutlierPropagationGenerator` | **yes** (fitted) | **yes** — descendants re-evaluated | appends perturbed + propagated rows |

* Outliers are an *appending* deficit: `Xt` is larger than `X`, and the
  generated rows are the **last** `n_outliers` rows.
* Outlier labels come from `get_outliers_labels()` — they do not ride
  in the deprecated `yt` slot.
* Part A's outliers are marginal extremes; Part B's outliers are
  structurally consistent extreme events whose footprint depends on
  the causal topology (chain / fork / collider / diamond).


## Takeaways

- Outlier generators add rows beyond the data's distribution; the output has MORE rows than the input.
- `generate` returns `(Xt, y)` with `y` a pass-through; the outlier flags come from `gen.get_outliers_labels()`.
- The causal variant (`CausalOutlierPropagationGenerator`) injects outliers at graph nodes and propagates them to descendants (Pearl surgery).